## Latent space cluster eval (E3 + E4) — VAE (`vae_v3`) — Yang & Lerch + UMAP→K-means

Self-contained Kaggle notebook that re-runs the latent-space evaluations
(E3 visualization, E4 neighborhood + interpolation) using **objective musical
similarity clusters** from Yang & Lerch 2020 features instead of the model's
conditioning attribute bins.

**Step 1:** Yang & Lerch 2020 feature fingerprints on the test set (~232-D).
**Step 2:** Z-score features; optional `IsolationForest` outlier pre-filter (**UMAP + K-means fit on inliers only**, all rows labeled via `transform` / `predict`). UMAP dimension sweep (10/20/50/100) → K-means in UMAP space; pick best dim by silhouette.

### Required Kaggle datasets (under `/kaggle/input/datasets/karandeepshoker/`)
- `vae_v3/ckpt.pt`
- `preprocessed-memmaptest/`
- `misc/tokenizer_meta.json` and `misc/quantiles.json`

### Outputs (under `/kaggle/working/artifacts/latent_eval/vae_v3_yl_umap/`)
All numerical results saved as `.npz` / `.npy` / `.json`. **Plotting is done
locally afterward** (no matplotlib in Kaggle).



In [ ]:
from __future__ import annotations
import os
import json
import math
import time
from pathlib import Path
from typing import Any, Callable, Optional
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
print('cwd:', os.getcwd())
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('torch cuda:', torch.version.cuda)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
INPUT_ROOT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working')
if INPUT_ROOT.exists():
    print('--- /kaggle/input ---')
    for p in sorted(INPUT_ROOT.iterdir()):
        if p.is_dir():
            print(' -', p.name)


In [ ]:
# -------------------------
# Config
# -------------------------
KAGGLE_USER = 'karandeepshoker'
MODEL_RUN = 'vae_v3'           # checkpoint folder under /kaggle/input/datasets/<user>/
MODEL_LABEL = 'vae_v3_yl_umap'       # used for output dir naming
TEST_DATASET = 'preprocessed-memmaptest'
TEST_SPLIT_REL_CANDIDATES = [Path('test'), Path('.')]
MISC_DATASET = 'misc'
TOKENIZER_META_NAME = 'tokenizer_meta.json'
QUANTILES_NAME = 'quantiles.json'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT_DIR = WORKING_ROOT / 'artifacts' / 'latent_eval' / MODEL_LABEL
OUT_DIR.mkdir(parents=True, exist_ok=True)
S1_BATCH_SIZE = 64
S1_N_TEST = 20_000             # cap test set used (None / 0 = all)

# Step 2 (K-means clustering)
S2_K_GRID = [4, 6, 8, 10, 12]
S2_SIL_SUBSAMPLE = 5_000
S2_KMEANS_N_INIT = 10
S2_RANDOM_STATE = 0

# Step 2 outlier pre-filter (UMAP/K-means fit on inliers only; all rows labeled via transform/predict)
S2_OUTLIER_METHOD = 'isolation_forest'  # 'isolation_forest' | 'l2_norm' | 'none'
S2_OUTLIER_CONTAMINATION = 0.02
S2_OUTLIER_N_ESTIMATORS = 100
S2_OUTLIER_L2_QUANTILE = 0.99  # used when S2_OUTLIER_METHOD == 'l2_norm'

# Step 2: UMAP dim sweep -> K-means in UMAP space (Yang & Lerch ~232-D)
S2_UMAP_KM_N_COMPONENTS_GRID = [10, 20, 50, 100]
S2_UMAP_KM_N_NEIGHBORS = 30
S2_UMAP_KM_MIN_DIST = 0.10
S2_UMAP_KM_METRIC = 'euclidean'

# Step 3 (feature-space PCA / UMAP)
UMAP_N_NEIGHBORS = 25
UMAP_MIN_DIST = 0.10

# Step 4 (E3: latent encode + kNN purity + NMI/ARI)
E3_BATCH_SIZE = 64
E3_KNN_K = 15

# Step 5 (E4: cluster-preservation neighborhood + interpolation)
E4_ANCHORS_PER_CLUSTER = 2
E4_SIGMAS = [0.10, 0.25, 0.50, 1.00]
E4_N_NEAR = 20
E4_N_PAIRS = 10
E4_N_INTERP = 9
E4_BASELINE_BIN = 3
E4_TEMPERATURE = 1.0
E4_TOP_P = 0.9
E4_SEED = 0
E4_TIME_BUDGET_SECS = int(3.5 * 3600)  # leave ~30 min headroom on a 4 h GPU run
print('OUT_DIR:', OUT_DIR)
print('DEVICE:', DEVICE)
print('MODEL_RUN:', MODEL_RUN)


In [ ]:
# -------------------------
# Locate dataset paths (mirrors notebook 08)

# -------------------------

def require_dir(p: Path) -> Path:
    if not p.exists() or not p.is_dir():
        raise FileNotFoundError(f'missing directory: {p}')
    return p

def require_file(p: Path) -> Path:
    if not p.exists() or not p.is_file():
        raise FileNotFoundError(f'missing file: {p}')
    return p
BASE = require_dir(Path('/kaggle/input/datasets') / KAGGLE_USER)
MODEL_CKPT = require_file(require_dir(BASE / MODEL_RUN) / 'ckpt.pt')
MISC_DIR = require_dir(BASE / MISC_DATASET)
TOKENIZER_META_PATH = require_file(MISC_DIR / TOKENIZER_META_NAME)
QUANTILES_PATH = require_file(MISC_DIR / QUANTILES_NAME)
TEST_ROOT = require_dir(BASE / TEST_DATASET)
TEST_DIR = None
for rel in TEST_SPLIT_REL_CANDIDATES:
    cand = TEST_ROOT / rel
    if cand.exists() and (cand / 'meta.json').exists():
        TEST_DIR = cand
        break
if TEST_DIR is None:
    raise FileNotFoundError(f'Could not find test split in {TEST_ROOT} (tried {TEST_SPLIT_REL_CANDIDATES})')
for f in ['meta.json', 'X.u16.memmap', 'bar_indices.u8.memmap', 'attributes.u8.memmap']:
    require_file(Path(TEST_DIR) / f)
print('MODEL_CKPT:', MODEL_CKPT)
print('TEST_DIR:', TEST_DIR)
print('TOKENIZER_META_PATH:', TOKENIZER_META_PATH)
print('QUANTILES_PATH:', QUANTILES_PATH)


In [ ]:
# -------------------------
# Self-contained module file (written to /kaggle/working/)
# Includes: model code, memmap dataset, grammar masking, top-p sampling,
# Includes: model code, memmap dataset, grammar masking, top-p sampling,
# Includes: model code, memmap dataset, grammar masking, top-p sampling,
# Includes: model code, memmap dataset, grammar masking, top-p sampling,
# Includes: model code, memmap dataset, grammar masking, top-p sampling,
# REMI generation, attribute recompute, and Yang & Lerch fingerprint.

# -------------------------
import sys
MODULE_PATH = Path('musicgen_kaggle_latent_min.py')
PLAIN_MODEL_CODE = r'''
from __future__ import annotations
from dataclasses import dataclass
from typing import Optional
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass(frozen=True)
class PlainTransformerConfig:
    vocab_size: int = 195
    block_size: int = 1024
    n_layers: int = 6
    n_embed: int = 512
    n_head: int = 8
    d_ff: int = 2048
    bars_per_sample: int = 8
    dropout: float = 0.1
    # Attributes: 4 dims (polyphony, intensity, velocity, density), each binned 0..7
    num_attributes: int = 4
    attribute_bins: int = 8
    attribute_embed_dim: int = 32  # 4*32 = 128
class AttributeConditioner(nn.Module):
    """
    attributes: LongTensor [B, bars, 4] with values in 0..7
    returns:    FloatTensor [B, bars, n_embed]
    """

    def __init__(self, cfg: PlainTransformerConfig):
        super().__init__()
        assert cfg.num_attributes == 4
        assert cfg.attribute_embed_dim * cfg.num_attributes == 128
        self.embeds = nn.ModuleList(
            [nn.Embedding(cfg.attribute_bins, cfg.attribute_embed_dim) for _ in range(cfg.num_attributes)]
        )
        self.proj = nn.Linear(cfg.attribute_embed_dim * cfg.num_attributes, cfg.n_embed)

    def forward(self, attributes: torch.Tensor) -> torch.Tensor:
        # attributes: [B, bars, 4]
        a0 = self.embeds[0](attributes[..., 0])
        a1 = self.embeds[1](attributes[..., 1])
        a2 = self.embeds[2](attributes[..., 2])
        a3 = self.embeds[3](attributes[..., 3])
        A = torch.cat([a0, a1, a2, a3], dim=-1)  # [B, bars, 128]
        return self.proj(A)  # [B, bars, n_embed]
class PlainCausalSelfAttention(nn.Module):

    def __init__(self, cfg: PlainTransformerConfig):
        super().__init__()
        assert cfg.n_embed % cfg.n_head == 0
        self.n_head = cfg.n_head
        self.head_dim = cfg.n_embed // cfg.n_head
        self.qkv = nn.Linear(cfg.n_embed, 3 * cfg.n_embed, bias=True)
        self.proj = nn.Linear(cfg.n_embed, cfg.n_embed, bias=True)
        self.dropout = float(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, C]
        B, T, C = x.shape
        qkv = self.qkv(x)  # [B, T, 3C]
        q, k, v = qkv.split(C, dim=-1)
        # [B, nh, T, hs]
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        # PyTorch uses optimized SDPA under the hood when available.
        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
        )  # [B, nh, T, hs]
        y = y.transpose(1, 2).contiguous().view(B, T, C)  # [B, T, C]
        return self.proj(y)
class PlainFiLMBlock(nn.Module):

    def __init__(self, cfg: PlainTransformerConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embed)
        self.attn = PlainCausalSelfAttention(cfg)
        self.resid_drop = nn.Dropout(cfg.dropout)
        # Conditioned FiLM: gamma/beta per token timestep, derived from cond_tok [B,T,C]
        self.film = nn.Linear(cfg.n_embed, 2 * cfg.n_embed, bias=True)
        self.ln2 = nn.LayerNorm(cfg.n_embed)
        self.mlp = nn.Sequential(
            nn.Linear(cfg.n_embed, cfg.d_ff),
            nn.GELU(),
            nn.Linear(cfg.d_ff, cfg.n_embed),
        )
        self.mlp_drop = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor, cond_tok: torch.Tensor) -> torch.Tensor:
        # Pre-norm
        h = self.ln1(x)
        gamma_beta = self.film(cond_tok)
        gamma, beta = gamma_beta.chunk(2, dim=-1)
        h = h * (1.0 + gamma) + beta
        x = x + self.resid_drop(self.attn(h))
        x = x + self.mlp_drop(self.mlp(self.ln2(x)))
        return x
class PlainTransformerDecoder(nn.Module):
    """Decoder-only baseline conditioned ONLY on per-bar attributes via FiLM."""

    def __init__(self, cfg: PlainTransformerConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embed)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embed)
        self.attr = AttributeConditioner(cfg)
        self.blocks = nn.ModuleList([PlainFiLMBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.n_embed)
        self.head = nn.Linear(cfg.n_embed, cfg.vocab_size, bias=True)

    def forward(
        self,
        X: torch.Tensor,
        bar_indices: torch.Tensor,
        attributes: torch.Tensor,
        *,
        targets: Optional[torch.Tensor] = None,
        pad_id: Optional[int] = None,
    ) -> dict:
        B, T = X.shape
        if T > self.cfg.block_size:
            raise ValueError(f"Sequence length T={T} exceeds block_size={self.cfg.block_size}")
        # Token + position embeddings
        pos = torch.arange(T, device=X.device)
        x = self.tok_emb(X) + self.pos_emb(pos)[None, :, :]
        # Build per-bar conditions and broadcast to tokens using bar_indices
        cond_bars = self.attr(attributes)  # [B, bars, C]
        idx = bar_indices.to(dtype=torch.long).clamp_(0, self.cfg.bars_per_sample - 1)
        cond_tok = cond_bars.gather(1, idx.unsqueeze(-1).expand(-1, -1, self.cfg.n_embed))  # [B, T, C]
        for blk in self.blocks:
            x = blk(x, cond_tok)
        x = self.ln_f(x)
        logits = self.head(x)  # [B, T, vocab]
        out = {"logits": logits}
        if targets is not None:
            if pad_id is None:
                raise ValueError("pad_id is required when computing loss with targets.")
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=int(pad_id),
                reduction="mean",
            )
            n_tokens = int((targets != int(pad_id)).sum().item())
            out.update({"loss": loss, "n_tokens": n_tokens})
        return out
'''
VAE_MODEL_CODE = r'''
from dataclasses import dataclass
import math
from typing import Optional
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass(frozen=True)
class VAEConfig:
    vocab_size: int = 195
    block_size: int = 1024
    # Encoder (bidirectional)
    enc_n_layers: int = 6
    enc_n_embed: int = 512
    enc_n_head: int = 8
    enc_d_ff: int = 2048
    z_dim: int = 128
    enc_dropout: float = 0.1
    # Attributes
    bars_per_sample: int = 8
    num_attributes: int = 4
    attribute_bins: int = 8
    attribute_embed_dim: int = 32  # 4*32 = 128
    # Conductor
    cond_hidden: int = 384
    cond_layers: int = 2
    # Decoder (causal)
    dec_n_layers: int = 6
    dec_n_embed: int = 512
    dec_n_head: int = 8
    dec_d_ff: int = 2048
    dec_dropout: float = 0.1
class AttributeEmbedder(nn.Module):

    def __init__(self, cfg: VAEConfig):
        super().__init__()
        assert cfg.num_attributes == 4
        self.embeds = nn.ModuleList(
            [nn.Embedding(cfg.attribute_bins, cfg.attribute_embed_dim) for _ in range(cfg.num_attributes)]
        )

    def forward(self, attributes: torch.Tensor) -> torch.Tensor:
        a0 = self.embeds[0](attributes[..., 0])
        a1 = self.embeds[1](attributes[..., 1])
        a2 = self.embeds[2](attributes[..., 2])
        a3 = self.embeds[3](attributes[..., 3])
        return torch.cat([a0, a1, a2, a3], dim=-1)

def broadcast_by_bar_indices(bars_x_c: torch.Tensor, bar_indices: torch.Tensor) -> torch.Tensor:
    B, bars, C = bars_x_c.shape
    idx = bar_indices.to(dtype=torch.long).clamp_(0, bars - 1)
    return bars_x_c.gather(1, idx.unsqueeze(-1).expand(B, idx.size(1), C))
class BiTransformerEncoder(nn.Module):

    def __init__(self, cfg: VAEConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.enc_n_embed)
        self.pos_emb = nn.Embedding(cfg.block_size + 1, cfg.enc_n_embed)
        self.cls = nn.Parameter(torch.zeros(cfg.enc_n_embed))
        layer = nn.TransformerEncoderLayer(
            d_model=cfg.enc_n_embed,
            nhead=cfg.enc_n_head,
            dim_feedforward=cfg.enc_d_ff,
            dropout=float(cfg.enc_dropout),
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=cfg.enc_n_layers)
        self.ln_f = nn.LayerNorm(cfg.enc_n_embed)
        self.mu = nn.Linear(cfg.enc_n_embed, cfg.z_dim)
        self.logvar = nn.Linear(cfg.enc_n_embed, cfg.z_dim)
        nn.init.normal_(self.cls, mean=0.0, std=0.02)

    def forward(self, X: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        B, T = X.shape
        if T != self.cfg.block_size:
            raise ValueError(f"Encoder expects block_size={self.cfg.block_size}, got T={T}")
        tok = self.tok_emb(X)
        cls = self.cls.view(1, 1, -1).expand(B, 1, -1)
        h = torch.cat([cls, tok], dim=1)
        pos = torch.arange(T + 1, device=X.device)
        h = h + self.pos_emb(pos)[None, :, :]
        h = self.encoder(h)
        h = self.ln_f(h)
        h_cls = h[:, 0, :]
        return self.mu(h_cls), self.logvar(h_cls)
class Conductor(nn.Module):

    def __init__(self, cfg: VAEConfig):
        super().__init__()
        self.cfg = cfg
        self.h0_proj = nn.Linear(cfg.z_dim, cfg.cond_layers * cfg.cond_hidden)
        self.bar_emb = nn.Embedding(cfg.bars_per_sample, cfg.cond_hidden)
        self.gru = nn.GRU(
            input_size=cfg.cond_hidden,
            hidden_size=cfg.cond_hidden,
            num_layers=cfg.cond_layers,
            batch_first=True,
        )

    def forward(self, z_p: torch.Tensor) -> torch.Tensor:
        B = z_p.size(0)
        h0 = self.h0_proj(z_p).view(self.cfg.cond_layers, B, self.cfg.cond_hidden).contiguous()
        bar_ids = torch.arange(self.cfg.bars_per_sample, device=z_p.device)
        x = self.bar_emb(bar_ids)[None, :, :].expand(B, -1, -1)
        z_seq, _hn = self.gru(x, h0)
        return z_seq
class CausalSelfAttention(nn.Module):

    def __init__(self, n_embed: int, n_head: int):
        super().__init__()
        assert n_embed % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embed // n_head
        self.qkv = nn.Linear(n_embed, 3 * n_embed, bias=True)
        self.proj = nn.Linear(n_embed, n_embed, bias=True)
        self.dropout = 0.0

    def set_dropout(self, p: float) -> None:
        self.dropout = float(p)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)
class FiLMBlock(nn.Module):

    def __init__(self, n_embed: int, n_head: int, d_ff: int, *, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embed)
        self.attn = CausalSelfAttention(n_embed=n_embed, n_head=n_head)
        self.attn.set_dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        self.film = nn.Linear(n_embed, 2 * n_embed, bias=True)
        self.ln2 = nn.LayerNorm(n_embed)
        self.mlp = nn.Sequential(
            nn.Linear(n_embed, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, n_embed),
        )
        self.mlp_drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, cond_tok: torch.Tensor) -> torch.Tensor:
        h = self.ln1(x)
        gamma, beta = self.film(cond_tok).chunk(2, dim=-1)
        h = h * (1.0 + gamma) + beta
        x = x + self.resid_drop(self.attn(h))
        x = x + self.mlp_drop(self.mlp(self.ln2(x)))
        return x
class FiLMDecoder(nn.Module):

    def __init__(self, cfg: VAEConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.dec_n_embed)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.dec_n_embed)
        self.blocks = nn.ModuleList(
            [
                FiLMBlock(cfg.dec_n_embed, cfg.dec_n_head, cfg.dec_d_ff, dropout=float(cfg.dec_dropout))
                for _ in range(cfg.dec_n_layers)
            ]
        )
        self.ln_f = nn.LayerNorm(cfg.dec_n_embed)
        self.head = nn.Linear(cfg.dec_n_embed, cfg.vocab_size, bias=True)

    def forward(self, X: torch.Tensor, cond_tok: torch.Tensor) -> torch.Tensor:
        B, T = X.shape
        if T != self.cfg.block_size:
            raise ValueError(f"Decoder expects block_size={self.cfg.block_size}, got T={T}")
        pos = torch.arange(T, device=X.device)
        x = self.tok_emb(X) + self.pos_emb(pos)[None, :, :]
        for blk in self.blocks:
            x = blk(x, cond_tok)
        x = self.ln_f(x)
        return self.head(x)
class MusicVAE(nn.Module):

    def __init__(self, cfg: VAEConfig):
        super().__init__()
        self.cfg = cfg
        self.attr = AttributeEmbedder(cfg)
        self.encoder = BiTransformerEncoder(cfg)
        self.conductor = Conductor(cfg)
        self.decoder = FiLMDecoder(cfg)
        if cfg.cond_hidden + (cfg.attribute_embed_dim * cfg.num_attributes) != cfg.dec_n_embed:
            raise ValueError(
                f"Expected cond_hidden(={cfg.cond_hidden}) + attr_dim(={cfg.attribute_embed_dim*cfg.num_attributes}) == dec_n_embed(={cfg.dec_n_embed})"
            )

    @staticmethod

    def reparameterize(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    @staticmethod

    def kl_divergence_free_bits(
        mu: torch.Tensor,
        logvar: torch.Tensor,
        *,
        free_bits_lambda_bits: float = 1.0,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        kl_per_dim = -0.5 * (1.0 + logvar - mu.pow(2) - logvar.exp())
        kl_per_dim_mean = kl_per_dim.mean(dim=0)
        lambda_nats = float(free_bits_lambda_bits) * math.log(2.0)
        kl_per_dim_hinged_mean = torch.clamp(kl_per_dim_mean, min=lambda_nats)
        kl_loss = kl_per_dim_hinged_mean.sum()
        active_units = (kl_per_dim_mean > lambda_nats).sum()
        return kl_loss, kl_per_dim_mean, active_units

    def forward(
        self,
        X: torch.Tensor,
        bar_indices: torch.Tensor,
        attributes: torch.Tensor,
        *,
        z_p: Optional[torch.Tensor] = None,
        targets: Optional[torch.Tensor] = None,
        pad_id: Optional[int] = None,
        beta: float = 1.0,
    ) -> dict:
        mu: Optional[torch.Tensor]
        logvar: Optional[torch.Tensor]
        if z_p is None:
            mu, logvar = self.encoder(X)
            z_p = self.reparameterize(mu, logvar)

        else:
            mu, logvar = None, None
            if z_p.ndim != 2 or z_p.shape[0] != X.shape[0]:
                raise ValueError(f"z_p must have shape [B,z_dim], got {tuple(z_p.shape)}")
            if z_p.shape[1] != self.cfg.z_dim:
                raise ValueError(f"z_p dim mismatch: expected {self.cfg.z_dim}, got {z_p.shape[1]}")
        z_k = self.conductor(z_p)
        A_k = self.attr(attributes)
        C_k = torch.cat([z_k, A_k], dim=-1)
        cond_tok = broadcast_by_bar_indices(C_k, bar_indices)
        logits = self.decoder(X, cond_tok)
        out = {"logits": logits, "mu": mu, "logvar": logvar, "z_p": z_p, "z_k": z_k}
        if mu is not None and logvar is not None:
            loss_kl, kl_per_dim_mean, active_units = self.kl_divergence_free_bits(mu, logvar, free_bits_lambda_bits=1.0)
            out["loss_kl"] = loss_kl
            out["kl_per_dim_mean"] = kl_per_dim_mean
            out["active_units"] = active_units
        if targets is not None:
            if pad_id is None:
                raise ValueError("pad_id is required when computing reconstruction loss.")
            if mu is None or logvar is None:
                raise ValueError("targets provided but z_p was passed explicitly; KL stats are undefined in this mode.")
            loss_recon = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=int(pad_id),
                reduction="mean",
            )
            out["loss_recon"] = loss_recon
            out["loss_total"] = loss_recon + float(beta) * loss_kl
        return out
'''
DATA_CODE = r'''
import json
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Optional
import numpy as np
import torch
from torch.utils.data import Dataset
PathLike = str | os.PathLike[str]

@dataclass(frozen=True)
class MemmapSplitMeta:
    split: str
    num_examples: int
    tokens_per_example: int
    bars_per_sample: int
    num_attributes: int
    pad_id: int
    bar_id: int
class MemmapMusicDataset(Dataset):

    def __init__(self, split_dir: PathLike):
        self.split_dir = Path(split_dir)
        payload = json.loads((self.split_dir / 'meta.json').read_text(encoding='utf-8'))
        self.meta = MemmapSplitMeta(
            split=str(payload.get('split', self.split_dir.name)),
            num_examples=int(payload['num_examples']),
            tokens_per_example=int(payload.get('tokens_per_example', payload.get('block_size', 1024))),
            bars_per_sample=int(payload.get('bars_per_sample', 8)),
            num_attributes=4,
            pad_id=int(payload['token_ids']['pad_id']),
            bar_id=int(payload['token_ids']['bar_id']),
        )
        self.x_path = self.split_dir / 'X.u16.memmap'
        self.bi_path = self.split_dir / 'bar_indices.u8.memmap'
        self.a_path = self.split_dir / 'attributes.u8.memmap'
        self._pid: Optional[int] = None
        self._x = None
        self._bi = None
        self._a = None

    def __len__(self) -> int:
        return self.meta.num_examples

    def _ensure_open(self) -> None:
        pid = os.getpid()
        if self._pid == pid and self._x is not None:
            return
        n = self.meta.num_examples
        t = self.meta.tokens_per_example
        b = self.meta.bars_per_sample
        a = self.meta.num_attributes
        self._x = np.memmap(self.x_path, mode='r', dtype=np.uint16, shape=(n, t))
        self._bi = np.memmap(self.bi_path, mode='r', dtype=np.uint8, shape=(n, t))
        self._a = np.memmap(self.a_path, mode='r', dtype=np.uint8, shape=(n, b, a))
        self._pid = pid

    def __getitem__(self, idx: int):
        self._ensure_open()
        return self._x[idx], self._bi[idx], self._a[idx]

    def __len__(self) -> int:
        return self.meta.num_examples

    def _ensure_open(self) -> None:
        pid = os.getpid()
        if self._pid == pid and self._x is not None:
            return
        n = self.meta.num_examples
        t = self.meta.tokens_per_example
        b = self.meta.bars_per_sample
        a = self.meta.num_attributes
        self._x = np.memmap(self.x_path, mode='r', dtype=np.uint16, shape=(n, t))
        self._bi = np.memmap(self.bi_path, mode='r', dtype=np.uint8, shape=(n, t))
        self._a = np.memmap(self.a_path, mode='r', dtype=np.uint8, shape=(n, b, a))
        self._pid = pid

    def __getitem__(self, idx: int):
        self._ensure_open()
        return self._x[idx], self._bi[idx], self._a[idx]

def make_collate_fn(*, pad_id: int) -> Callable:

    def collate(batch):
        xs, bis, attrs = zip(*batch)
        x_np = np.stack(xs, axis=0)
        bi_np = np.stack(bis, axis=0)
        a_np = np.stack(attrs, axis=0)
        X = torch.from_numpy(x_np.astype(np.int64, copy=False))
        bar_indices = torch.from_numpy(bi_np.astype(np.int64, copy=False))
        attributes = torch.from_numpy(a_np.astype(np.int64, copy=False))
        Y = torch.empty_like(X)
        Y[:, :-1] = X[:, 1:]
        Y[:, -1] = int(pad_id)
        return {'X': X, 'Y': Y, 'bar_indices': bar_indices, 'attributes': attributes}
    return collate
'''
GRAMMAR_CODE = r'''
from dataclasses import dataclass
from typing import Iterable, Literal
import torch
TokenType = Literal[
    'PAD',
    'BOS',
    'Bar',
    'Position',
    'Tempo',
    'Pitch',
    'Velocity',
    'Duration',
    'Other',
]

def token_type(token_str: str) -> TokenType:
    head = token_str.split('_', 1)[0]
    if head == 'PAD':
        return 'PAD'
    if head == 'BOS':
        return 'BOS'
    if head == 'Bar':
        return 'Bar'
    if head == 'Position':
        return 'Position'
    if head == 'Tempo':
        return 'Tempo'
    if head == 'Pitch':
        return 'Pitch'
    if head == 'Velocity':
        return 'Velocity'
    if head == 'Duration':
        return 'Duration'
    return 'Other'

@dataclass(frozen=True)
class GrammarVocabIndex:
    ids_by_type: dict[TokenType, list[int]]

def build_vocab_index(id_to_token: dict[int, str]) -> GrammarVocabIndex:
    ids_by_type: dict[TokenType, list[int]] = {
        'PAD': [],
        'BOS': [],
        'Bar': [],
        'Position': [],
        'Tempo': [],
        'Pitch': [],
        'Velocity': [],
        'Duration': [],
        'Other': [],
    }
    for tid, s in id_to_token.items():
        ids_by_type[token_type(s)].append(int(tid))
    for k in ids_by_type:
        ids_by_type[k] = sorted(ids_by_type[k])
    return GrammarVocabIndex(ids_by_type=ids_by_type)

def allowed_next_ids(
    prev_token_id: int,
    *,
    id_to_token: dict[int, str],
    vocab_index: GrammarVocabIndex,
    bar_id: int,
    pad_id: int,
    bos_id: int,
    allow_bar: bool = True,
) -> list[int]:
    prev_str = id_to_token[int(prev_token_id)]
    prev_type = token_type(prev_str)
    if prev_token_id == int(bos_id) or prev_type == 'BOS':
        return [int(bar_id)]
    if prev_token_id == int(bar_id) or prev_type == 'Bar':
        return vocab_index.ids_by_type['Position']
    if prev_type == 'Pitch':
        return vocab_index.ids_by_type['Velocity']
    if prev_type == 'Velocity':
        return vocab_index.ids_by_type['Duration']
    if prev_type == 'Tempo':
        return vocab_index.ids_by_type['Position']
    if prev_type == 'Position':
        allowed: list[int] = []
        allowed += vocab_index.ids_by_type['Pitch']
        allowed += vocab_index.ids_by_type['Tempo']
        if allow_bar:
            allowed.append(int(bar_id))
        return sorted(set(allowed))
    if prev_type == 'Duration':
        allowed = []
        allowed += vocab_index.ids_by_type['Pitch']
        allowed += vocab_index.ids_by_type['Position']
        allowed += vocab_index.ids_by_type['Tempo']
        if allow_bar:
            allowed.append(int(bar_id))
        return sorted(set(allowed))
    allowed = []
    allowed += vocab_index.ids_by_type['Bar'] if allow_bar else []
    allowed += vocab_index.ids_by_type['Position']
    allowed += vocab_index.ids_by_type['Tempo']
    allowed += vocab_index.ids_by_type['Pitch']
    allowed = [i for i in allowed if i not in (int(pad_id), int(bos_id))]
    return sorted(set(allowed))

def mask_logits_inplace(
    logits: torch.Tensor,
    *,
    allowed_ids: Iterable[int],
) -> torch.Tensor:
    if logits.ndim != 1:
        raise ValueError(f'Expected logits [vocab], got {tuple(logits.shape)}')
    allowed = torch.tensor(list(allowed_ids), device=logits.device, dtype=torch.long)
    mask = torch.ones((logits.size(0),), device=logits.device, dtype=torch.bool)
    mask[allowed] = False
    logits[mask] = float('-inf')
    return logits
'''
SAMPLE_CODE = r'''
from typing import Optional
import torch

def sample_next_token(
    logits: torch.Tensor,
    *,
    top_p: float = 0.9,
    temperature: float = 1.0,
    generator: Optional[torch.Generator] = None,
) -> int:
    if logits.ndim != 1:
        raise ValueError(f'Expected logits shape [vocab], got {tuple(logits.shape)}')
    if not (0.0 < float(top_p) <= 1.0):
        raise ValueError('top_p must be in (0, 1].')
    if float(temperature) <= 0.0:
        raise ValueError('temperature must be > 0.')
    if float(temperature) != 1.0:
        logits = logits / float(temperature)
    probs = torch.softmax(logits - logits.max(), dim=-1)
    sorted_probs, sorted_idx = torch.sort(probs, descending=True)
    cdf = torch.cumsum(sorted_probs, dim=-1)
    keep = cdf <= float(top_p)
    keep[0] = True
    if keep.sum().item() < keep.numel():
        first_excluded = int(keep.sum().item())
        keep[first_excluded] = True
    filt_probs = sorted_probs[keep]
    filt_idx = sorted_idx[keep]
    filt_probs = filt_probs / filt_probs.sum()
    choice = torch.multinomial(filt_probs, num_samples=1, replacement=False, generator=generator)
    next_id = int(filt_idx[int(choice.item())].item())
    return next_id
'''
ATTR_CODE = r'''
import json
from pathlib import Path
import numpy as np
ATTRIBUTE_NAMES = ['polyphony_rate', 'rhythmic_intensity', 'velocity_dynamics', 'note_density']
BLOCK_SIZE = 1024
BARS_PER_SAMPLE = 8

def _parse_duration_beats(token: str) -> float:
    raw = token.split('_', 1)[1]
    try:
        return float(raw)
    except Exception:
        parts = raw.split('.')
        if len(parts) == 3:
            a, b, c = int(parts[0]), int(parts[1]), int(parts[2])
            if c == 0:
                raise ValueError(f'Cannot parse duration token: {token}')
            return float(a + (b / c))
        raise ValueError(f'Cannot parse duration token: {token}')

def compute_raw_attributes_per_bar_from_tokens(
    x_tokens_1024: list[int],
    *,
    id_to_token: dict[int, str],
    bar_token_id: int,
    bars_per_sample: int = BARS_PER_SAMPLE,
) -> list[list[float]]:
    if len(x_tokens_1024) != BLOCK_SIZE:
        raise ValueError(f'Expected {BLOCK_SIZE} tokens, got {len(x_tokens_1024)}')
    onsets = [0 for _ in range(bars_per_sample)]
    pos_has_onset: list[set[int]] = [set() for _ in range(bars_per_sample)]
    velocities: list[list[int]] = [[] for _ in range(bars_per_sample)]
    global_intervals: list[tuple[float, float]] = []
    cur_bar = -1
    cur_pos = 0

    def parse_pos(tok: str) -> int:
        return int(tok.split('_', 1)[1])

    def parse_vel(tok: str) -> int:
        return int(tok.split('_', 1)[1])
    i = 0
    while i < len(x_tokens_1024):
        tid = x_tokens_1024[i]
        tok = id_to_token.get(int(tid))
        if tok is None:
            i += 1
            continue
        typ = tok.split('_', 1)[0]
        if tid == bar_token_id or typ == 'Bar':
            cur_bar += 1
            cur_pos = 0
            i += 1
            continue
        if cur_bar < 0:
            i += 1
            continue
        if cur_bar >= bars_per_sample:
            break
        if typ == 'Position':
            cur_pos = parse_pos(tok)
            i += 1
            continue
        if typ == 'Pitch' and i + 2 < len(x_tokens_1024):
            vtok = id_to_token.get(int(x_tokens_1024[i + 1]), '')
            dtok = id_to_token.get(int(x_tokens_1024[i + 2]), '')
            if vtok.startswith('Velocity_') and dtok.startswith('Duration_'):
                vel = parse_vel(vtok)
                dur = _parse_duration_beats(dtok)
                onsets[cur_bar] += 1
                pos_has_onset[cur_bar].add(cur_pos)
                velocities[cur_bar].append(vel)
                s = cur_bar * 4.0 + (cur_pos * 0.25)
                e = s + dur
                global_intervals.append((s, e))
                i += 3
                continue
        i += 1
    attrs: list[list[float]] = []
    for b in range(bars_per_sample):
        note_density = float(onsets[b])
        rhythmic_intensity = float(len(pos_has_onset[b]) / 16.0)
        vel_std = float(np.std(np.asarray(velocities[b], dtype=float), ddof=0)) if velocities[b] else 0.0
        poly_samples: list[int] = []
        for p in range(16):
            t = b * 4.0 + (p * 0.25)
            active = 0
            for s, e in global_intervals:
                if s <= t < e:
                    active += 1
            poly_samples.append(active)
        polyphony_rate = float(sum(poly_samples) / 16.0)
        attrs.append([polyphony_rate, rhythmic_intensity, vel_std, note_density])
    return attrs

def bin_attributes(
    raw_attrs_barsx4: list[list[float]],
    thresholds: dict[str, list[float]],
    bars_per_sample: int = BARS_PER_SAMPLE,
) -> list[list[int]]:
    arr = np.asarray(raw_attrs_barsx4, dtype=float)
    if arr.ndim != 2 or arr.shape != (bars_per_sample, 4):
        raise ValueError(f'Expected shape [{bars_per_sample},4], got {arr.shape}')
    out = np.zeros_like(arr, dtype=int)
    for j, name in enumerate(ATTRIBUTE_NAMES):
        th = np.asarray(thresholds[name], dtype=float)
        if th.shape != (7,):
            raise ValueError(f'Expected 7 thresholds for {name}, got shape {th.shape}')
        out[:, j] = np.digitize(arr[:, j], th, right=False)
    return out.astype(int).tolist()

def load_thresholds(path: str) -> dict[str, list[float]]:
    payload = json.loads(Path(path).read_text(encoding='utf-8'))
    th = payload['thresholds']
    for name in ATTRIBUTE_NAMES:
        if name not in th:
            raise ValueError(f'Threshold for {name!r} missing in {path}')
    return th
'''
GENERATE_CODE = r'''
from dataclasses import dataclass
from typing import Literal, Optional
import torch
from musicgen_kaggle_latent_min import build_vocab_index, allowed_next_ids, mask_logits_inplace
from musicgen_kaggle_latent_min import sample_next_token
ModelType = Literal['plain', 'vae', 'simple_vae']

@dataclass(frozen=True)
class GenerationConfig:
    block_size: int = 1024
    bars_per_sample: int = 8
    temperature: float = 1.0
    top_p: float = 0.9

def compute_bar_indices_prefix(tokens: list[int], *, bar_id: int, bars_per_sample: int) -> list[int]:
    cur = -1
    max_bar = bars_per_sample - 1
    out: list[int] = []
    for tid in tokens:
        if int(tid) == int(bar_id):
            cur += 1
        if cur < 0:
            cur = 0
        if cur > max_bar:
            cur = max_bar
        out.append(int(cur))
    return out

@torch.no_grad()

def generate_remi_tokens(
    *,
    model_type: ModelType,
    model,
    attributes: torch.Tensor,
    pad_id: int,
    bos_id: int,
    bar_id: int,
    id_to_token: dict[int, str],
    cfg: GenerationConfig,
    device: torch.device,
    z_p: Optional[torch.Tensor] = None,
    seed: Optional[int] = None,
    vocab_index=None,
) -> list[int]:
    if attributes.ndim != 3 or attributes.shape[0] != 1 or attributes.shape[1] != cfg.bars_per_sample or attributes.shape[2] != 4:
        raise ValueError(f'attributes must have shape [1,{cfg.bars_per_sample},4], got {tuple(attributes.shape)}')
    model.eval()
    model.to(device)
    gen = torch.Generator(device=device)
    if seed is not None:
        gen.manual_seed(int(seed))
    if vocab_index is None:
        vocab_index = build_vocab_index(id_to_token)
    tokens: list[int] = [int(bos_id)]
    bar_count = 0
    X_full = torch.full((1, cfg.block_size), int(pad_id), dtype=torch.long, device=device)
    z_p_in = None
    if model_type in ('vae', 'simple_vae'):
        if z_p is None:
            z_p_in = torch.randn((1, getattr(model.cfg, 'z_dim', 128)), generator=gen, device=device)

        else:
            z_p_in = z_p.to(device=device, dtype=torch.float32)
            if z_p_in.ndim != 2 or z_p_in.shape[0] != 1:
                raise ValueError(f'z_p must have shape [1,z_dim], got {tuple(z_p_in.shape)}')
    for _t in range(cfg.block_size - 1):
        X_full.fill_(int(pad_id))
        X_full[0, : len(tokens)] = torch.tensor(tokens, dtype=torch.long, device=device)
        bar_idx_prefix = compute_bar_indices_prefix(tokens, bar_id=bar_id, bars_per_sample=cfg.bars_per_sample)
        cur_bar = int(bar_idx_prefix[-1]) if bar_idx_prefix else 0
        bar_indices = torch.full((1, cfg.block_size), cur_bar, dtype=torch.long, device=device)
        bar_indices[0, : len(bar_idx_prefix)] = torch.tensor(bar_idx_prefix, dtype=torch.long, device=device)
        if model_type == 'plain':
            out = model(X_full, bar_indices=bar_indices, attributes=attributes)
        elif model_type == 'simple_vae':
            out = model(X_full, bar_indices=bar_indices, attributes=attributes, z_p=z_p_in)
        elif model_type == 'vae':
            out = model(X_full, bar_indices=bar_indices, attributes=attributes, z_p=z_p_in)

        else:
            raise ValueError(f'Unknown model_type: {model_type}')
        logits = out['logits']
        step_logits = logits[0, len(tokens) - 1].float().clone()
        prev_id = int(tokens[-1])
        allowed = allowed_next_ids(
            prev_id,
            id_to_token=id_to_token,
            vocab_index=vocab_index,
            bar_id=bar_id,
            pad_id=pad_id,
            bos_id=bos_id,
            allow_bar=True,
        )
        mask_logits_inplace(step_logits, allowed_ids=allowed)
        next_id = sample_next_token(step_logits, top_p=cfg.top_p, temperature=cfg.temperature, generator=gen)
        if int(next_id) == int(bar_id) and bar_count >= cfg.bars_per_sample:
            break
        tokens.append(int(next_id))
        if int(next_id) == int(bar_id):
            bar_count += 1
    if len(tokens) < cfg.block_size:
        tokens = tokens + [int(pad_id)] * (cfg.block_size - len(tokens))

    else:
        tokens = tokens[: cfg.block_size]
    return tokens
'''
YL_CODE = r'''
import numpy as np
# Length-bin edges (in beats). Yang & Lerch use 8 bins. Our REMI Duration tokens
# encode beat values in {0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0, ...}; the bins
# below cover this range.
NOTE_LENGTH_BIN_EDGES = np.array([0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0], dtype=np.float64)
# 8 bins -> [<=0.25), [0.25-0.5), [0.5-0.75), [0.75-1.0), [1.0-1.5), [1.5-2.0), [2.0-3.0), [>=3.0)
N_LEN_BINS = 8
YL_FEATURE_NAMES = [
    'pitch_range',                       # 1
    'n_used_pitch_classes',              # 1
    *[f'pc_hist_{i}' for i in range(12)], # 12
    *[f'pc_trans_{i}_{j}' for i in range(12) for j in range(12)],  # 144
    'note_count',                        # 1
    'avg_ioi_beats',                     # 1
    *[f'len_hist_{i}' for i in range(N_LEN_BINS)],  # 8
    *[f'len_trans_{i}_{j}' for i in range(N_LEN_BINS) for j in range(N_LEN_BINS)],  # 64
]
YL_FEATURE_DIM = len(YL_FEATURE_NAMES)  # 232

def _build_token_lookup(id_to_token: dict[int, str], vocab_size: int):
    """Return parallel arrays for fast O(1) token decoding from id -> (type_code, value).
    type_code: 0=Other, 1=Bar, 2=Position, 3=Pitch, 4=Velocity, 5=Duration, 6=Tempo, 7=PAD/BOS
    value: parsed numeric payload (int for Position/Pitch/Velocity; float beats for Duration; -1 otherwise)
    """
    type_arr = np.zeros((vocab_size,), dtype=np.int8)
    val_arr = np.full((vocab_size,), -1.0, dtype=np.float64)
    for tid, s in id_to_token.items():
        tid = int(tid)
        if not (0 <= tid < vocab_size):
            continue
        head, _, tail = s.partition('_')
        if head == 'Bar':
            type_arr[tid] = 1
        elif head == 'Position':
            type_arr[tid] = 2
            try:
                val_arr[tid] = int(tail)
            except Exception:
                pass
        elif head == 'Pitch':
            type_arr[tid] = 3
            try:
                val_arr[tid] = int(tail)
            except Exception:
                pass
        elif head == 'Velocity':
            type_arr[tid] = 4
            try:
                val_arr[tid] = int(tail)
            except Exception:
                pass
        elif head == 'Duration':
            type_arr[tid] = 5
            try:
                val_arr[tid] = float(tail)
            except Exception:
                # handle "a.b.c" style fractional encodings
                parts = tail.split('.')
                if len(parts) == 3:
                    a, b, c = int(parts[0]), int(parts[1]), int(parts[2])
                    if c != 0:
                        val_arr[tid] = float(a + (b / c))
        elif head == 'Tempo':
            type_arr[tid] = 6
        elif head in ('PAD', 'BOS'):
            type_arr[tid] = 7

        else:
            type_arr[tid] = 0
    return type_arr, val_arr

def yang_lerch_fingerprint(
    tokens: np.ndarray,
    *,
    type_arr: np.ndarray,
    val_arr: np.ndarray,
    bars_per_sample: int = 8,
) -> np.ndarray:
    """Compute one ~232-D Yang & Lerch fingerprint from a [T] sequence of REMI token ids.
    Operates per-song (not per-bar). Skips PAD/BOS, and ignores Pitch tokens whose
    immediately following Velocity / Duration partner is missing or malformed.
    """
    if tokens.ndim != 1:
        raise ValueError(f'tokens must be 1-D, got shape {tuple(tokens.shape)}')
    T = int(tokens.shape[0])
    pitches: list[int] = []
    durations: list[float] = []
    onset_beats: list[float] = []  # for IOI
    cur_bar = -1
    cur_pos = 0  # 0..15 ticks
    i = 0
    while i < T:
        tid = int(tokens[i])
        ty = int(type_arr[tid])
        if ty == 7:  # PAD/BOS
            i += 1
            continue
        if ty == 1:  # Bar
            cur_bar += 1
            cur_pos = 0
            i += 1
            continue
        if cur_bar < 0:
            i += 1
            continue
        if cur_bar >= bars_per_sample:
            break
        if ty == 2:  # Position
            cur_pos = int(val_arr[tid])
            i += 1
            continue
        if ty == 6:  # Tempo
            i += 1
            continue
        if ty == 3 and i + 2 < T:  # Pitch + Velocity + Duration triple
            tid_v = int(tokens[i + 1])
            tid_d = int(tokens[i + 2])
            if int(type_arr[tid_v]) == 4 and int(type_arr[tid_d]) == 5:
                pitch = int(val_arr[tid])
                dur = float(val_arr[tid_d])
                if pitch >= 0 and dur > 0:
                    pitches.append(pitch)
                    durations.append(dur)
                    onset_beats.append(float(cur_bar) * 4.0 + float(cur_pos) * 0.25)
                i += 3
                continue
        i += 1
    feat = np.zeros((YL_FEATURE_DIM,), dtype=np.float64)
    if len(pitches) == 0:
        return feat.astype(np.float32)
    pitches_arr = np.asarray(pitches, dtype=np.int64)
    pcs = pitches_arr % 12
    feat[0] = float(pitches_arr.max() - pitches_arr.min())  # pitch range
    feat[1] = float(len(np.unique(pcs)))                     # # used pitch classes
    pc_hist = np.bincount(pcs, minlength=12).astype(np.float64)
    if pc_hist.sum() > 0:
        pc_hist /= pc_hist.sum()
    feat[2 : 2 + 12] = pc_hist
    pc_trans = np.zeros((12, 12), dtype=np.float64)
    for a, b in zip(pcs[:-1], pcs[1:]):
        pc_trans[int(a), int(b)] += 1.0
    if pc_trans.sum() > 0:
        pc_trans /= pc_trans.sum()
    feat[14 : 14 + 144] = pc_trans.reshape(-1)
    feat[14 + 144] = float(len(pitches_arr))  # note count
    if len(onset_beats) >= 2:
        ob = np.asarray(onset_beats, dtype=np.float64)
        ob.sort()
        iois = np.diff(ob)
        iois = iois[iois > 0]
        if iois.size > 0:
            feat[14 + 144 + 1] = float(iois.mean())
    durs_arr = np.asarray(durations, dtype=np.float64)
    len_bins = np.digitize(durs_arr, NOTE_LENGTH_BIN_EDGES, right=False)  # 0..N_LEN_BINS-1
    len_bins = np.clip(len_bins, 0, N_LEN_BINS - 1)
    len_hist = np.bincount(len_bins, minlength=N_LEN_BINS).astype(np.float64)
    if len_hist.sum() > 0:
        len_hist /= len_hist.sum()
    base = 14 + 144 + 2
    feat[base : base + N_LEN_BINS] = len_hist
    len_trans = np.zeros((N_LEN_BINS, N_LEN_BINS), dtype=np.float64)
    for a, b in zip(len_bins[:-1], len_bins[1:]):
        len_trans[int(a), int(b)] += 1.0
    if len_trans.sum() > 0:
        len_trans /= len_trans.sum()
    feat[base + N_LEN_BINS : base + N_LEN_BINS + N_LEN_BINS * N_LEN_BINS] = len_trans.reshape(-1)
    return feat.astype(np.float32)

def yang_lerch_batch(
    X: np.ndarray,
    *,
    id_to_token: dict[int, str],
    vocab_size: int,
    bars_per_sample: int = 8,
) -> np.ndarray:
    """Compute fingerprints for a batch X of shape [N, T]. Returns float32 [N, 232]."""
    if X.ndim != 2:
        raise ValueError(f'X must be 2-D, got shape {tuple(X.shape)}')
    type_arr, val_arr = _build_token_lookup(id_to_token, vocab_size)
    out = np.zeros((X.shape[0], YL_FEATURE_DIM), dtype=np.float32)
    for i in range(X.shape[0]):
        out[i] = yang_lerch_fingerprint(
            np.asarray(X[i], dtype=np.int64),
            type_arr=type_arr,
            val_arr=val_arr,
            bars_per_sample=bars_per_sample,
        )
    return out
'''
MODULE_CODE = (
    PLAIN_MODEL_CODE
    + "\n\n"
    + VAE_MODEL_CODE
    + "\n\n"
    + DATA_CODE
    + "\n\n"
    + GRAMMAR_CODE
    + "\n\n"
    + SAMPLE_CODE
    + "\n\n"
    + ATTR_CODE
    + "\n\n"
    + GENERATE_CODE
    + "\n\n"
    + YL_CODE
)
MODULE_PATH.write_text(MODULE_CODE, encoding='utf-8')
print('Wrote module:', MODULE_PATH, '({} bytes)'.format(MODULE_PATH.stat().st_size))
if str(Path('.').resolve()) not in sys.path:
    sys.path.insert(0, str(Path('.').resolve()))
# Force a fresh import in case a stale module from a previous run lingers.
if 'musicgen_kaggle_latent_min' in sys.modules:
    del sys.modules['musicgen_kaggle_latent_min']
import musicgen_kaggle_latent_min as mg
print('Imported self-contained module:', mg.__file__)
print('YL_FEATURE_DIM =', mg.YL_FEATURE_DIM)


In [ ]:
# -------------------------
# Load tokenizer metadata + thresholds

# -------------------------
tok_meta = json.loads(TOKENIZER_META_PATH.read_text(encoding='utf-8'))
if isinstance(tok_meta['id_to_token'], dict):
    id_to_token = {int(k): v for k, v in tok_meta['id_to_token'].items()}

else:
    id_to_token = {int(i): s for i, s in enumerate(tok_meta['id_to_token'])}
pad_id = int(tok_meta.get('pad_id', -1))
bos_id = int(tok_meta.get('bos_id', -1))
bar_id = int(tok_meta.get('bar_id', -1))
if pad_id < 0 or bos_id < 0 or bar_id < 0:
    meta_json = json.loads((Path(TEST_DIR) / 'meta.json').read_text(encoding='utf-8'))
    pad_id = int(meta_json['token_ids']['pad_id'])
    bos_id = int(meta_json['token_ids']['bos_id'])
    bar_id = int(meta_json['token_ids']['bar_id'])
thresholds = json.loads(QUANTILES_PATH.read_text(encoding='utf-8'))['thresholds']
vocab_index = mg.build_vocab_index(id_to_token)
VOCAB_SIZE = max(id_to_token.keys()) + 1
print('pad_id:', pad_id, 'bos_id:', bos_id, 'bar_id:', bar_id)
print('vocab size:', VOCAB_SIZE)
print('built vocab_index')


In [ ]:
# -------------------------
# DataLoader + memmap dataset

# -------------------------
from torch.utils.data import DataLoader
test_ds = mg.MemmapMusicDataset(TEST_DIR)
collate_fn = mg.make_collate_fn(pad_id=pad_id)
test_dl = DataLoader(
    test_ds,
    batch_size=S1_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    drop_last=False,
    collate_fn=collate_fn,
)
print('test meta:', test_ds.meta)
print('total examples:', len(test_ds))


In [ ]:
# -------------------------
# Load checkpoint + instantiate model

# -------------------------
MODEL_TYPE = 'vae'

def _load_ckpt(path):
    ckpt = torch.load(path, map_location='cpu')
    if isinstance(ckpt, dict) and 'cfg' in ckpt:
        return ckpt
    raise ValueError(f'Unexpected ckpt format at {path}')
ckpt = _load_ckpt(MODEL_CKPT)
cfg_obj = mg.VAEConfig(**ckpt['cfg']) if isinstance(ckpt['cfg'], dict) else ckpt['cfg']
model = mg.MusicVAE(cfg_obj)
sd = ckpt.get('model') or ckpt.get('model_state_dict') or ckpt.get('state_dict')
if sd is None:
    raise KeyError('ckpt missing model state dict')
model.load_state_dict(sd, strict=True)
model.to(DEVICE)
model.eval()
Z_DIM = int(getattr(model.cfg, 'z_dim', 128))
print('loaded model:', MODEL_TYPE, '- cfg:', getattr(model, 'cfg', None))
print('z_dim:', Z_DIM)
# Quick sanity: 1 batch -> 1 forward pass.
b = next(iter(test_dl))
X_s = b['X'].to(DEVICE)
Y_s = b['Y'].to(DEVICE)
bi_s = b['bar_indices'].to(DEVICE)
attr_s = b['attributes'].to(DEVICE)
with torch.no_grad():
    out_s = model(X_s, bar_indices=bi_s, attributes=attr_s, targets=Y_s, pad_id=pad_id)
print('sanity logits:', tuple(out_s['logits'].shape),
      'loss_recon:', float(out_s['loss_recon']),
      'loss_kl:', float(out_s['loss_kl']))


In [ ]:
# -------------------------
# Helpers (progress + slerp + standardization reuse + nearest-centroid)

# -------------------------
PRINT_EVERY = 1

def _fmt_secs(s: float) -> str:
    s = max(0.0, float(s))
    if s < 60:
        return f"{s:5.1f}s"
    m = s / 60.0
    if m < 60:
        return f"{m:5.1f}m"
    h = m / 60.0
    return f"{h:5.2f}h"

def print_progress(*, tag: str, i: int, total: int, start_time: float, every: int = 1) -> None:
    if every <= 0:
        return
    if i % every != 0 and i != total:
        return
    now = time.time()
    elapsed = now - start_time
    rate = elapsed / max(1, i)
    eta = rate * max(0, total - i)
    print(f"[{tag}] {i}/{total}  elapsed={_fmt_secs(elapsed)}  eta={_fmt_secs(eta)}")

def slerp(t: float, a: np.ndarray, b: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    a = a.astype(np.float64)
    b = b.astype(np.float64)
    na = np.linalg.norm(a) + eps
    nb = np.linalg.norm(b) + eps
    a_u = a / na
    b_u = b / nb
    dot = float(np.clip(np.dot(a_u, b_u), -1.0, 1.0))
    omega = math.acos(dot)
    if abs(omega) < 1e-6:
        out = (1.0 - t) * a + t * b

    else:
        so = math.sin(omega)
        out = (math.sin((1.0 - t) * omega) / so) * a_u + (math.sin(t * omega) / so) * b_u
        # Re-scale to interpolated norm so we don't shrink the latent.
        out = out * ((1.0 - t) * na + t * nb)
    return out

def standardize_with_scaler(F_raw: np.ndarray, *, keep_mask: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    """Apply the saved Step-2 scaler: drop dead columns + z-score with epsilon."""
    if F_raw.ndim == 1:
        F_raw = F_raw[None, :]
    F_kept = F_raw[:, keep_mask]
    return (F_kept - mean) / (std + 1e-8)

def assign_nearest_centroid(F_std: np.ndarray, centroids: np.ndarray) -> np.ndarray:
    """Euclidean nearest-centroid assignment. F_std: [N, D], centroids: [K, D]."""
    if F_std.ndim == 1:
        F_std = F_std[None, :]
    d2 = ((F_std[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=-1)
    return d2.argmin(axis=1).astype(np.int64)


In [ ]:
# -------------------------

# Step 1 - Yang & Lerch 2020 feature fingerprints over the test set

# -------------------------
# Pre-build token-type / value lookup once (re-used for every batch).
type_arr, val_arr = mg._build_token_lookup(id_to_token, VOCAB_SIZE)
n_total = len(test_ds)
if S1_N_TEST and S1_N_TEST > 0:
    n_total = min(n_total, int(S1_N_TEST))
F_list = []
n_done = 0
start = time.time()
total_batches = int(math.ceil(n_total / S1_BATCH_SIZE))
for bi, batch in enumerate(tqdm(test_dl, desc='Step1 fingerprint', total=total_batches), start=1):
    Xb = batch['X'].cpu().numpy().astype(np.int64)
    bs = Xb.shape[0]
    if n_done + bs > n_total:
        Xb = Xb[: n_total - n_done]
        bs = Xb.shape[0]
    feats = np.zeros((bs, mg.YL_FEATURE_DIM), dtype=np.float32)
    for k in range(bs):
        feats[k] = mg.yang_lerch_fingerprint(
            Xb[k],
            type_arr=type_arr,
            val_arr=val_arr,
            bars_per_sample=int(test_ds.meta.bars_per_sample),
        )
    F_list.append(feats)
    n_done += bs
    print_progress(tag='Step1', i=bi, total=total_batches, start_time=start, every=PRINT_EVERY)
    if n_done >= n_total:
        break
F = np.concatenate(F_list, axis=0)[:n_total]
print('F shape:', F.shape, 'dtype:', F.dtype)
print('NaN count in F:', int(np.isnan(F).sum()))
np.savez_compressed(OUT_DIR / 'feature_fingerprints.npz', F=F)
print('Wrote', OUT_DIR / 'feature_fingerprints.npz')


In [ ]:
# -------------------------

# Step 2 - UMAP dim sweep (10/20/50/100) + K-means in UMAP space (IsolationForest outlier pre-filter)
#   * z-score Yang & Lerch F_std; IsolationForest on F_std; UMAP+K-means on inliers; transform/predict all rows
#   * sweep UMAP n_components; k-grid per dim in UMAP space
#   * pick best n_components by silhouette at chosen k; final fit on inlier rows
#   * persist scaler + umap_kmeans_reducer for E4 (transform fp_std -> nearest centroid)

# -------------------------
import joblib
import umap
from collections import Counter
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
mean_full = F.mean(axis=0)
std_full = F.std(axis=0)
keep_mask = std_full > 1e-8
print(f'kept {int(keep_mask.sum())} / {F.shape[1]} columns (dropped {int((~keep_mask).sum())} zero-variance cols)')
mean_kept = mean_full[keep_mask]
std_kept = std_full[keep_mask]
F_std = (F[:, keep_mask] - mean_kept) / (std_kept + 1e-8)
assert np.isfinite(F_std).all(), 'F_std contains NaN/Inf after standardization'
F_all = F_std.astype(np.float32)
outlier_guard_fallback = False
n = int(F_std.shape[0])
method = str(S2_OUTLIER_METHOD).lower().strip()

if method in ('', 'none'):
    inlier_mask = np.ones(n, dtype=bool)

elif method == 'isolation_forest':
    from sklearn.ensemble import IsolationForest
    iso = IsolationForest(
        n_estimators=int(S2_OUTLIER_N_ESTIMATORS),
        contamination=float(S2_OUTLIER_CONTAMINATION),
        random_state=int(S2_RANDOM_STATE),
        n_jobs=-1,
    )
    iso.fit(F_std)
    inlier_mask = iso.predict(F_std) == 1

elif method == 'l2_norm':
    row_norm = np.linalg.norm(F_std.astype(np.float64), axis=1)
    thr = float(np.quantile(row_norm, float(S2_OUTLIER_L2_QUANTILE)))
    inlier_mask = row_norm <= thr

else:
    raise ValueError(f'Unknown S2_OUTLIER_METHOD: {S2_OUTLIER_METHOD!r} (use isolation_forest | l2_norm | none)')
n_in = int(inlier_mask.sum())
n_out = int(n - n_in)
print(
    f'outlier prefilter method={method!r} inliers={n_in} outliers={n_out} '
    f'({n_out / max(1, n):.4f} frac outliers)'
)
min_required = max(1000, int(S2_SIL_SUBSAMPLE), 2 * max(int(k) for k in S2_K_GRID))
if n_in < min_required:
    print(f'WARN: inliers {n_in} < required {min_required}; disabling outlier filter for this run')
    inlier_mask = np.ones(n, dtype=bool)
    outlier_guard_fallback = True
    n_in = int(inlier_mask.sum())
    n_out = int(n - n_in)
F_in = F_std[inlier_mask]
print('UMAP/K-means fit rows:', F_in.shape[0], ' x ', F_in.shape[1])
n_samples, d_kept = int(F_std.shape[0]), int(F_std.shape[1])
rng_sil = np.random.default_rng(int(S2_RANDOM_STATE))
inlier_idx = np.flatnonzero(inlier_mask)
sub_n = int(min(S2_SIL_SUBSAMPLE, inlier_idx.size))
sub_idx = rng_sil.choice(inlier_idx, size=sub_n, replace=False)
print(f'silhouette / CH / DB subsample: n={sub_n} (inliers only, UMAP space, per dim sweep)')
dim_sweep = {}

def _pick_k_from_scores(sil_scores, ch_scores, db_scores):
    k_by_sil = int(max(sil_scores, key=lambda kk: sil_scores[kk]))
    k_by_ch = int(max(ch_scores, key=lambda kk: ch_scores[kk]))
    k_by_db = int(min(db_scores, key=lambda kk: db_scores[kk]))
    votes = Counter([k_by_sil, k_by_ch, k_by_db])
    top_k, top_votes = votes.most_common(1)[0]
    if top_votes >= 2:
        k_best = int(top_k)
        rule = f'majority vote (k={k_best} got {top_votes}/3 votes)'

    else:
        k_best = int(k_by_ch)
        rule = f'three-way tie -> Calinski-Harabasz winner (k={k_best})'
    return k_best, rule, k_by_sil, k_by_ch, k_by_db, votes

for target_n in S2_UMAP_KM_N_COMPONENTS_GRID:
    n_umap = min(int(target_n), d_kept, max(2, n_samples - 2))
    nn_km = int(min(S2_UMAP_KM_N_NEIGHBORS, max(2, n_samples - 1)))
    print(f'\n---- UMAP target={target_n} -> n_components={n_umap} ----')
    start = time.time()
    reducer = umap.UMAP(
        n_components=int(n_umap),
        n_neighbors=nn_km,
        min_dist=float(S2_UMAP_KM_MIN_DIST),
        metric=str(S2_UMAP_KM_METRIC),
        random_state=int(S2_RANDOM_STATE),
    )
    reducer.fit(F_all[inlier_mask])
    E_km = reducer.transform(F_all)
    print(f'E_km shape={E_km.shape} elapsed={_fmt_secs(time.time() - start)}')
    F_sub = E_km[sub_idx]
    sil_scores_d = {}
    ch_scores_d = {}
    db_scores_d = {}
    start = time.time()

    for ki, k in enumerate(S2_K_GRID, start=1):
        km = KMeans(n_clusters=int(k), n_init=int(S2_KMEANS_N_INIT), random_state=int(S2_RANDOM_STATE))
        lab = km.fit_predict(F_sub)
        sil_scores_d[int(k)] = float(silhouette_score(F_sub, lab))
        ch_scores_d[int(k)] = float(calinski_harabasz_score(F_sub, lab))
        db_scores_d[int(k)] = float(davies_bouldin_score(F_sub, lab))
        print_progress(
            tag=f'  dim={n_umap} k={k:>3d}  sil={sil_scores_d[int(k)]:+.4f}',
            i=ki, total=len(S2_K_GRID), start_time=start, every=PRINT_EVERY,
        )
    k_best_d, rule_d, k_by_sil_d, k_by_ch_d, k_by_db_d, votes_d = _pick_k_from_scores(
        sil_scores_d, ch_scores_d, db_scores_d
    )
    sil_at_k = float(sil_scores_d[int(k_best_d)])
    dim_sweep[int(target_n)] = {
        'target_components': int(target_n),
        'umap_n_components': int(n_umap),
        'chosen_k': int(k_best_d),
        'selection_rule': rule_d,
        'silhouette_at_chosen_k': sil_at_k,
        'silhouette_per_k': sil_scores_d,
        'calinski_harabasz_per_k': ch_scores_d,
        'davies_bouldin_per_k': db_scores_d,
        'winner_silhouette': int(k_by_sil_d),
        'winner_calinski_harabasz': int(k_by_ch_d),
        'winner_davies_bouldin': int(k_by_db_d),
        'votes': {int(k): int(v) for k, v in votes_d.items()},
    }
    print(f'  chosen k={k_best_d}  sil@k={sil_at_k:+.4f}  rule={rule_d}')

# Pick UMAP dimension with best silhouette at its chosen k.
best_target = int(max(dim_sweep, key=lambda t: dim_sweep[t]['silhouette_at_chosen_k']))
best_entry = dim_sweep[best_target]
n_umap = int(best_entry['umap_n_components'])
umap_selection_rule = (
    f'best silhouette@chosen_k among grid {list(S2_UMAP_KM_N_COMPONENTS_GRID)} '
    f'-> target={best_target} (n_components={n_umap}, sil={best_entry["silhouette_at_chosen_k"]:+.4f})'
)
print('\n---- UMAP dimension selection ----')
print(umap_selection_rule)

# Final UMAP + K-means at selected dimension.
start = time.time()
nn_km = int(min(S2_UMAP_KM_N_NEIGHBORS, max(2, n_samples - 1)))
reducer_km = umap.UMAP(
    n_components=int(n_umap),
    n_neighbors=nn_km,
    min_dist=float(S2_UMAP_KM_MIN_DIST),
    metric=str(S2_UMAP_KM_METRIC),
    random_state=int(S2_RANDOM_STATE),
)
reducer_km.fit(F_all[inlier_mask])
E_km = reducer_km.transform(F_all)
print(f'final E_km shape={E_km.shape} elapsed={_fmt_secs(time.time() - start)}')
joblib.dump(reducer_km, OUT_DIR / 'umap_kmeans_reducer.joblib')
sil_scores = best_entry['silhouette_per_k']
ch_scores = best_entry['calinski_harabasz_per_k']
db_scores = best_entry['davies_bouldin_per_k']
k_best = int(best_entry['chosen_k'])
selection_rule = best_entry['selection_rule']
k_by_sil = int(best_entry['winner_silhouette'])
k_by_ch = int(best_entry['winner_calinski_harabasz'])
k_by_db = int(best_entry['winner_davies_bouldin'])
votes = {int(k): int(v) for k, v in best_entry['votes'].items()}
print('---- Step 2 cluster validation (final UMAP dim) ----')
print(f'silhouette per k         : {sil_scores}   -> winner k={k_by_sil}')
print(f'Calinski-Harabasz per k  : {ch_scores}   -> winner k={k_by_ch}')
print(f'Davies-Bouldin per k     : {db_scores}   -> winner k={k_by_db}')
print(f'votes per k              : {votes}')
print(f'k selection rule         : {selection_rule}')
print(f'chosen k                 : {k_best}')
km_final = KMeans(n_clusters=k_best, n_init=int(S2_KMEANS_N_INIT), random_state=int(S2_RANDOM_STATE))
km_final.fit(E_km[inlier_mask])
cluster_labels = km_final.predict(E_km).astype(np.int64)
centroids = km_final.cluster_centers_.astype(np.float32)
print('cluster_labels:', cluster_labels.shape, 'centroids:', centroids.shape)
assert centroids.shape[1] == int(n_umap), (centroids.shape, n_umap)
np.save(OUT_DIR / 'cluster_labels.npy', cluster_labels)
np.save(OUT_DIR / 'centroids.npy', centroids)
np.save(OUT_DIR / 'inlier_mask.npy', inlier_mask.astype(np.bool_))
np.savez_compressed(
    OUT_DIR / 'feature_scaler.npz',
    keep_mask=keep_mask.astype(np.bool_),
    mean=mean_kept.astype(np.float32),
    std=std_kept.astype(np.float32),
)
(OUT_DIR / 'umap_dim_sweep.json').write_text(json.dumps(dim_sweep, indent=2), encoding='utf-8')
(OUT_DIR / 'chosen_k.json').write_text(
    json.dumps(
        {
            'clustering_method': 'umap_then_kmeans',
            'feature_type': 'yang_lerch_2020',
            'standardized': True,
            'outlier_method': method,
            'outlier_contamination': float(S2_OUTLIER_CONTAMINATION) if method == 'isolation_forest' else None,
            'outlier_l2_quantile': float(S2_OUTLIER_L2_QUANTILE) if method == 'l2_norm' else None,
            'outlier_n_estimators': int(S2_OUTLIER_N_ESTIMATORS) if method == 'isolation_forest' else None,
            'n_inliers': int(n_in),
            'n_outliers': int(n_out),
            'outlier_guard_fallback': bool(outlier_guard_fallback),
            'umap_kmeans_n_components': int(n_umap),
            'umap_kmeans_target_components': int(best_target),
            'umap_dim_selection_rule': umap_selection_rule,
            'umap_dim_sweep': dim_sweep,
            'k_grid': list(map(int, S2_K_GRID)),
            'silhouette_per_k': sil_scores,
            'calinski_harabasz_per_k': ch_scores,
            'davies_bouldin_per_k': db_scores,
            'winner_silhouette': int(k_by_sil),
            'winner_calinski_harabasz': int(k_by_ch),
            'winner_davies_bouldin': int(k_by_db),
            'votes': votes,
            'selection_rule': selection_rule,
            'chosen_k': int(k_best),
            'subsample_size': int(sub_n),
            'random_state': int(S2_RANDOM_STATE),
            'kmeans_n_init': int(S2_KMEANS_N_INIT),
            'umap_kmeans_n_neighbors': int(S2_UMAP_KM_N_NEIGHBORS),
            'umap_kmeans_min_dist': float(S2_UMAP_KM_MIN_DIST),
            'umap_kmeans_metric': str(S2_UMAP_KM_METRIC),
            'feature_dim_kept': int(d_kept),
        },
        indent=2,
    ),
    encoding='utf-8',
)
print('Wrote cluster_labels.npy, centroids.npy, inlier_mask.npy, umap_kmeans_reducer.joblib, feature_scaler.npz, umap_dim_sweep.json, chosen_k.json')


In [ ]:
# -------------------------

# Step 3 - Feature-space PCA-2D + UMAP-2D ("ground truth" view)
#   Save as feature_pca.npz / feature_umap.npz (no PNG; rendered locally).

# -------------------------
# PCA-2D via SVD on centered F_std (already mean-zeroed by construction).
F_center = F_std - F_std.mean(axis=0, keepdims=True)
start = time.time()
U, S, Vt = np.linalg.svd(F_center, full_matrices=False)
feature_pca2 = (U[:, :2] * S[:2]).astype(np.float32)
print(f'feature_pca2 shape={feature_pca2.shape} elapsed={_fmt_secs(time.time() - start)}')
np.savez_compressed(
    OUT_DIR / 'feature_pca.npz',
    pca2=feature_pca2,
    labels=cluster_labels.astype(np.int64),
    chosen_k=np.int64(k_best),
)
print('Wrote', OUT_DIR / 'feature_pca.npz')
# UMAP-2D
feature_umap2 = None
try:
    import umap
    start = time.time()
    um = umap.UMAP(
        n_components=2,
        n_neighbors=int(UMAP_N_NEIGHBORS),
        min_dist=float(UMAP_MIN_DIST),
        metric='euclidean',
        random_state=int(S2_RANDOM_STATE),
    )
    feature_umap2 = um.fit_transform(F_std).astype(np.float32)
    print(f'feature_umap2 shape={feature_umap2.shape} elapsed={_fmt_secs(time.time() - start)}')
except Exception as e:
    print('UMAP not available, skipping. Error:', repr(e))
if feature_umap2 is not None:
    np.savez_compressed(
        OUT_DIR / 'feature_umap.npz',
        umap2=feature_umap2,
        labels=cluster_labels.astype(np.int64),
        chosen_k=np.int64(k_best),
    )
    print('Wrote', OUT_DIR / 'feature_umap.npz')

else:
    print('Skipped feature_umap.npz')


In [ ]:
# -------------------------

# Step 4 - E3 (latent visualization + kNN purity + NMI/ARI)
#   * Encode mu for the same N test sequences used in Step 1
#   * PCA-2D + UMAP-2D on mu
#   * kNN purity (PCA / UMAP) wrt cluster_labels (local metric)
#   * Run K-means on mu with same k_best -> latent_labels, then NMI / ARI (global metric)

# -------------------------
from sklearn.cluster import KMeans as _KMeans
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from sklearn.neighbors import NearestNeighbors
vae_dl = DataLoader(
    test_ds,
    batch_size=E3_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    drop_last=False,
    collate_fn=collate_fn,
)
mus = []
n_seen = 0
start = time.time()
total_batches = int(math.ceil(F.shape[0] / E3_BATCH_SIZE))
pbar = tqdm(vae_dl, desc='Step4 encode mu', total=total_batches)
for bi, batch in enumerate(pbar, start=1):
    Xb = batch['X'].to(DEVICE)
    with torch.no_grad():
        mu, _logvar = model.encoder(Xb)
    mu_np = mu.detach().cpu().numpy()
    if n_seen + mu_np.shape[0] > F.shape[0]:
        mu_np = mu_np[: F.shape[0] - n_seen]
    mus.append(mu_np)
    n_seen += mu_np.shape[0]
    print_progress(tag='Step4 encode', i=bi, total=total_batches, start_time=start, every=PRINT_EVERY)
    if n_seen >= F.shape[0]:
        break
mu_all = np.concatenate(mus, axis=0)[: F.shape[0]]
print('mu_all:', mu_all.shape)
# PCA-2D on mu via SVD
mu_center = mu_all - mu_all.mean(axis=0, keepdims=True)
U, S, Vt = np.linalg.svd(mu_center, full_matrices=False)
mu_pca2 = (U[:, :2] * S[:2]).astype(np.float32)
# UMAP-2D on mu
mu_umap2 = None
try:
    import umap
    um = umap.UMAP(
        n_components=2,
        n_neighbors=int(UMAP_N_NEIGHBORS),
        min_dist=float(UMAP_MIN_DIST),
        metric='euclidean',
        random_state=int(S2_RANDOM_STATE),
    )
    mu_umap2 = um.fit_transform(mu_all).astype(np.float32)
except Exception as e:
    print('UMAP not available for mu, skipping. Error:', repr(e))

def knn_purity(emb: np.ndarray, lab: np.ndarray, *, k: int) -> float:
    nn = NearestNeighbors(n_neighbors=k + 1, algorithm='auto').fit(emb)
    _, idx = nn.kneighbors(emb)
    idx = idx[:, 1:]  # drop self
    neigh = lab[idx]
    return float((neigh == lab[:, None]).mean())
pur_pca = float(knn_purity(mu_pca2, cluster_labels, k=int(E3_KNN_K)))
pur_umap = float(knn_purity(mu_umap2, cluster_labels, k=int(E3_KNN_K))) if mu_umap2 is not None else None
print('kNN purity (PCA-2D)  =', pur_pca, '   random baseline =', 1.0 / k_best)
print('kNN purity (UMAP-2D) =', pur_umap)
# K-means on latent mu with same k_best -> latent_labels for NMI / ARI
km_lat = _KMeans(n_clusters=k_best, n_init=int(S2_KMEANS_N_INIT), random_state=int(S2_RANDOM_STATE))
latent_labels = km_lat.fit_predict(mu_all).astype(np.int64)
assert latent_labels.shape == cluster_labels.shape, (latent_labels.shape, cluster_labels.shape)
nmi = float(normalized_mutual_info_score(cluster_labels, latent_labels))
ari = float(adjusted_rand_score(cluster_labels, latent_labels))
print('NMI(latent_labels, cluster_labels) =', nmi)
print('ARI(latent_labels, cluster_labels) =', ari, '  (random ARI ~ 0)')
np.savez_compressed(OUT_DIR / 'e3_latent_mu.npz', mu=mu_all.astype(np.float32), labels=cluster_labels.astype(np.int64), pca2=mu_pca2)
if mu_umap2 is not None:
    np.savez_compressed(OUT_DIR / 'e3_umap.npz', umap2=mu_umap2)
np.save(OUT_DIR / 'latent_labels.npy', latent_labels)
(OUT_DIR / 'e3_knn_purity_clusters.json').write_text(
    json.dumps({
        'knn_purity_pca': pur_pca,
        'knn_purity_umap': pur_umap,
        'nmi': nmi,
        'ari': ari,
        'random_baselines': {'knn_purity': 1.0 / k_best, 'ari': 0.0},
        'k_best': int(k_best),
        'k_neighbors': int(E3_KNN_K),
    }, indent=2),
    encoding='utf-8',
)
print('Wrote e3_latent_mu.npz, e3_umap.npz (if UMAP), latent_labels.npy, e3_knn_purity_clusters.json')


In [ ]:
# -------------------------

# Step 5 - E4 (cluster-preservation neighborhood + interpolation)
#   * E4_ANCHORS_PER_CLUSTER * k_best anchors, sampled from each cluster
#   * Sweep sigmas; for each (anchor, sigma) sample N_NEAR neighbors -> P(same cluster)
#   * E4_N_PAIRS anchor pairs (distinct cluster origins) -> SLERP 9 points -> cluster trajectory
#   * Auto-downscale if measured sec/sample blows past E4_TIME_BUDGET_SECS

# -------------------------

# Load scaler + UMAP k-means reducer (centroids live in UMAP space).
scaler_npz = np.load(OUT_DIR / 'feature_scaler.npz')
KEEP_MASK = scaler_npz['keep_mask'].astype(bool)
MEAN_KEPT = scaler_npz['mean'].astype(np.float32)
STD_KEPT = scaler_npz['std'].astype(np.float32)
reducer_km = joblib.load(OUT_DIR / 'umap_kmeans_reducer.joblib')
CENTROIDS = np.load(OUT_DIR / 'centroids.npy').astype(np.float32)
assert CENTROIDS.shape[1] == int(reducer_km.n_components), (CENTROIDS.shape, int(reducer_km.n_components))

# Build a function to: encode one test sample's mu and recompute its base cluster.

@torch.no_grad()

def encode_sample(idx: int):
    xb = test_ds._x[int(idx)] if hasattr(test_ds, '_x') and test_ds._x is not None else None
    # Make sure memmaps are open
    test_ds._ensure_open()
    xb = test_ds._x[int(idx)]
    ab = test_ds._a[int(idx)]
    Xt = torch.from_numpy(np.asarray(xb, dtype=np.int64))[None, :].to(DEVICE)
    mu, _ = model.encoder(Xt)
    return mu[0].detach().cpu().numpy().astype(np.float32), np.asarray(ab, dtype=np.int64)

def gen_decode_and_assign(z_np: np.ndarray, attrs_8x4: np.ndarray, *, seed: int) -> tuple[int, np.ndarray]:
    """Decode latent z under given per-bar attributes, recompute fingerprint, assign cluster."""
    z_t = torch.from_numpy(z_np.astype(np.float32))[None, :]
    attrs_t = torch.from_numpy(attrs_8x4.astype(np.int64))[None, :, :].to(DEVICE)
    gen_cfg = mg.GenerationConfig(
        block_size=int(test_ds.meta.tokens_per_example),
        bars_per_sample=int(test_ds.meta.bars_per_sample),
        temperature=float(E4_TEMPERATURE),
        top_p=float(E4_TOP_P),
    )
    tokens = mg.generate_remi_tokens(
        model_type='vae',
        model=model,
        attributes=attrs_t,
        pad_id=int(pad_id),
        bos_id=int(bos_id),
        bar_id=int(bar_id),
        id_to_token=id_to_token,
        cfg=gen_cfg,
        device=DEVICE,
        z_p=z_t,
        seed=int(seed),
        vocab_index=vocab_index,
    )
    fp = mg.yang_lerch_fingerprint(
        np.asarray(tokens, dtype=np.int64),
        type_arr=type_arr,
        val_arr=val_arr,
        bars_per_sample=int(test_ds.meta.bars_per_sample),
    )
    fp_std = standardize_with_scaler(fp[None, :], keep_mask=KEEP_MASK, mean=MEAN_KEPT, std=STD_KEPT)
    e_km = reducer_km.transform(fp_std.astype(np.float32))
    cl = int(assign_nearest_centroid(e_km, CENTROIDS)[0])
    return cl, fp

# --- Pick anchors: E4_ANCHORS_PER_CLUSTER per cluster ---
rng = np.random.default_rng(int(E4_SEED))
anchors = []  # list of dict(idx=int, cluster=int, mu=np.ndarray, attrs=np.ndarray)

for c in range(k_best):
    pool = np.where(cluster_labels == c)[0]
    if pool.size == 0:
        continue
    take = min(int(E4_ANCHORS_PER_CLUSTER), pool.size)
    chosen = rng.choice(pool, size=take, replace=False)
    for idx in chosen:
        mu_a, attrs_a = encode_sample(int(idx))
        anchors.append({'idx': int(idx), 'cluster': int(c), 'mu': mu_a, 'attrs': attrs_a})
print(f'[E4] anchors picked: {len(anchors)} ({E4_ANCHORS_PER_CLUSTER} per cluster x {k_best} clusters)')

# --- Time budget probe (1 generation) ---
probe_a = anchors[0]
t0 = time.time()
_pc, _pf = gen_decode_and_assign(probe_a['mu'], probe_a['attrs'], seed=int(E4_SEED) + 999_999)
sec_per_gen = max(1e-3, time.time() - t0)
print(f'[E4] probe sec/gen={sec_per_gen:.3f}')
# Auto-downscale to fit E4_TIME_BUDGET_SECS:
n_near_eff = int(E4_N_NEAR)
n_pairs_eff = int(E4_N_PAIRS)
anchors_eff = list(anchors)

def _planned_gens():
    return len(anchors_eff) * len(E4_SIGMAS) * n_near_eff + n_pairs_eff * int(E4_N_INTERP)

while _planned_gens() * sec_per_gen > E4_TIME_BUDGET_SECS:
    if n_near_eff > 5:
        n_near_eff = max(5, n_near_eff // 2)
        continue
    if n_pairs_eff > 4:
        n_pairs_eff = max(4, n_pairs_eff // 2)
        continue
    if len(anchors_eff) > k_best:  # at least 1 per cluster
        anchors_eff = anchors_eff[: max(k_best, len(anchors_eff) // 2)]
        continue
    break
print(f'[E4] planned generations = {_planned_gens()} ({n_near_eff} near, {n_pairs_eff} pairs, {len(anchors_eff)} anchors)')

# --- Neighborhood: P(same cluster) vs sigma ---
near_records = []
total_units = len(anchors_eff) * len(E4_SIGMAS) * n_near_eff
unit = 0
start = time.time()

for ai, a in enumerate(tqdm(anchors_eff, desc='E4 neighborhood anchors')):
    base_cluster, base_fp = gen_decode_and_assign(a['mu'], a['attrs'], seed=int(E4_SEED) + 100_000 + ai)

    for sig in E4_SIGMAS:
        for j in range(n_near_eff):
            unit += 1
            print_progress(tag='E4 neigh', i=unit, total=total_units, start_time=start, every=PRINT_EVERY)
            eps = rng.standard_normal(int(Z_DIM)).astype(np.float32)
            z = a['mu'] + float(sig) * eps
            cl, _fp = gen_decode_and_assign(z, a['attrs'], seed=int(E4_SEED) + 200_000 + 10_000 * ai + 1_000 * E4_SIGMAS.index(sig) + j)
            near_records.append({
                'anchor_idx': int(a['idx']),
                'anchor_cluster': int(a['cluster']),
                'base_cluster': int(base_cluster),
                'sigma': float(sig),
                'sample': int(j),
                'l2_z': float(np.linalg.norm(z - a['mu'])),
                'cluster': int(cl),
                'same_as_base': int(int(cl) == int(base_cluster)),
            })

# --- Interpolation: SLERP between distinct-cluster anchor pairs ---
# Build all eligible (i, j) pairs with anchors[i].cluster != anchors[j].cluster
elig = [(i, j) for i in range(len(anchors_eff)) for j in range(len(anchors_eff)) if i != j and anchors_eff[i]['cluster'] != anchors_eff[j]['cluster']]
rng.shuffle(elig)
pairs = elig[: int(n_pairs_eff)]
interp_records = []
total_units = int(n_pairs_eff) * int(E4_N_INTERP)
unit = 0
start = time.time()

for pi, (i, j) in enumerate(tqdm(pairs, desc='E4 interpolation pairs')):
    a = anchors_eff[i]
    b = anchors_eff[j]

    for ti, t in enumerate(np.linspace(0.0, 1.0, int(E4_N_INTERP))):
        unit += 1
        print_progress(tag='E4 interp', i=unit, total=total_units, start_time=start, every=PRINT_EVERY)
        zt = slerp(float(t), a['mu'], b['mu']).astype(np.float32)
        cl, fp = gen_decode_and_assign(zt, a['attrs'], seed=int(E4_SEED) + 300_000 + 1000 * pi + ti)
        interp_records.append({
            'pair': [int(a['idx']), int(b['idx'])],
            'pair_clusters': [int(a['cluster']), int(b['cluster'])],
            't': float(t),
            'cluster': int(cl),
            'fingerprint': fp.astype(np.float32).tolist(),
        })
(OUT_DIR / 'e4_neighborhood_clusters.json').write_text(
    json.dumps({
        'sigmas': list(map(float, E4_SIGMAS)),
        'n_near_per_anchor_per_sigma': int(n_near_eff),
        'k_best': int(k_best),
        'records': near_records,
    }, indent=2),
    encoding='utf-8',
)
(OUT_DIR / 'e4_interp_clusters.json').write_text(
    json.dumps({
        'n_pairs': int(n_pairs_eff),
        'n_interp_steps': int(E4_N_INTERP),
        'k_best': int(k_best),
        'records': interp_records,
    }, indent=2),
    encoding='utf-8',
)
print('Wrote e4_neighborhood_clusters.json and e4_interp_clusters.json')


In [ ]:
# -------------------------

# Step 6 - Summary index

# -------------------------
_ck = json.loads((OUT_DIR / 'chosen_k.json').read_text(encoding='utf-8'))
summary = {
    'model_run': MODEL_RUN,
    'model_label': MODEL_LABEL,
    'model_type': MODEL_TYPE,
    'embedder': 'yang_lerch_2020',
    'standardized': True,
    'clustering_method': _ck.get('clustering_method'),
    'umap_kmeans_n_components': _ck.get('umap_kmeans_n_components'),
    'umap_kmeans_target_components': _ck.get('umap_kmeans_target_components'),
    'k_best': int(k_best),
    'k_grid': list(map(int, S2_K_GRID)),
    'silhouette_per_k': {int(k): float(v) for k, v in sil_scores.items()},
    'calinski_harabasz_per_k': {int(k): float(v) for k, v in ch_scores.items()},
    'davies_bouldin_per_k': {int(k): float(v) for k, v in db_scores.items()},
    'cluster_selection_rule': selection_rule,
    'cluster_selection_winners': {
        'silhouette': int(k_by_sil),
        'calinski_harabasz': int(k_by_ch),
        'davies_bouldin': int(k_by_db),
    },
    'n_test_used': int(F.shape[0]),
    'feature_dim_raw': int(F.shape[1]),
    'feature_dim_kept': int(int(keep_mask.sum())),
    'outlier_method': _ck.get('outlier_method'),
    'n_inliers': _ck.get('n_inliers'),
    'n_outliers': _ck.get('n_outliers'),
    'outlier_guard_fallback': _ck.get('outlier_guard_fallback'),
    'artifacts': {
        'feature_fingerprints': 'feature_fingerprints.npz',
        'feature_scaler': 'feature_scaler.npz',
        'umap_kmeans_reducer': 'umap_kmeans_reducer.joblib',
        'umap_dim_sweep': 'umap_dim_sweep.json',
        'feature_pca': 'feature_pca.npz',
        'feature_umap': 'feature_umap.npz' if (OUT_DIR / 'feature_umap.npz').exists() else None,
        'cluster_labels': 'cluster_labels.npy',
        'centroids': 'centroids.npy',
        'inlier_mask': 'inlier_mask.npy',
        'chosen_k': 'chosen_k.json',
        'e3_latent_mu': 'e3_latent_mu.npz',
        'e3_umap': 'e3_umap.npz' if (OUT_DIR / 'e3_umap.npz').exists() else None,
        'latent_labels': 'latent_labels.npy',
        'e3_knn_purity_clusters': 'e3_knn_purity_clusters.json',
        'e4_neighborhood_clusters': 'e4_neighborhood_clusters.json',
        'e4_interp_clusters': 'e4_interp_clusters.json',
    },
}
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Wrote', OUT_DIR / 'summary.json')
print('Done.')
